# 01-google-trends

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/mobility/trends/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('mobility/trends'))


In [ ]:
%pprint
import numpy as np 
import pandas as pd
import os
import time
import datetime
import requests
import csv
import json
import re

import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from tqdm import tqdm

%config InlineBackend.figure_format = 'retina'
import warnings
# Keep warnings visible when checking the research environment.

In [ ]:
# Locate folder
os.chdir(workspace('mobility/trends'))  

In [ ]:
# Goole groCode to Metro Name
dt = pd.read_json('google-trends-locations.json', typ='series')
cities = [ city for city in list(dt.index) if re.match(r'US-[A-Z][A-Z]-\d{3}', city)]

In [ ]:
dt[cities]

## Collect data

In [ ]:
from pytrends.request import TrendReq
pytrends = TrendReq(hl='en-US', tz=360)
pytrends.suggestions("coronavirus")

In [ ]:
%%time
# search by region
from pytrends.request import TrendReq
pytrends = TrendReq(hl='en-US', tz=360)
error = []

days = pd.date_range(start='2020-01-01', end='2020-09-01')
days = [d.strftime("%Y-%m-%d %Y-%m-%d") for d in days]

for day in tqdm(days):
    time.sleep(1)
    pytrends.build_payload(['/g/11j2cc_qll'], timeframe=day, geo='US')
    if day == days[0]:
        result_d = pytrends.interest_by_region(resolution='DMA', inc_low_vol=True, 
                                             inc_geo_code=True).rename(columns={'/g/11j2cc_qll':day[:10]})
    else:
        try: 
            result_d[day[:10]] = pytrends.interest_by_region(resolution='DMA', 
                                                             inc_low_vol=True, inc_geo_code=True)['/g/11j2cc_qll']
        except:
            error.append(day)
            continue

display(result_d)

In [ ]:
result_d.geoCode = pd.Series(dt[cities].index.values, index=dt[cities])
result_d = result_d.drop(columns=['geoCode_2'], errors='ignore').set_index('geoCode')
result_d
result_d.to_csv('google-trend-Coronavirus_disease_2019_byregion.csv')

In [ ]:
%%time
# search by day
from pytrends.request import TrendReq
pytrends = TrendReq(hl='en-US', tz=360)
error = []

for city in tqdm(cities):
    time.sleep(1)
    pytrends.build_payload(['/g/11j2cc_qll'], timeframe='2020-01-01 2020-09-01', geo=city)
    if city == cities[0]:
        result = pytrends.interest_over_time().drop(columns=['isPartial']).rename(columns={'/g/11j2cc_qll':city})
    else:
        try:
            temp = pytrends.interest_over_time().drop(columns=['isPartial']).rename(columns={'/g/11j2cc_qll':city})
            result = pd.concat([result, temp], axis=1)
        except:
            error.append(city)
            continue

display(result)

In [ ]:
result.loc[:,result.columns[:10]].plot(figsize=(15,8))

In [ ]:
result.to_csv('google-trend-Coronavirus_disease_2019_byday.csv')

## Sort

In [ ]:
by_day = pd.read_csv('google-trend-Coronavirus_disease_2019_byday.csv', index_col=0)
by_region = pd.read_csv('google-trend-Coronavirus_disease_2019_byregion.csv', index_col=0)

by_day = by_day.reindex(sorted(by_day.columns), axis=1)
by_region = by_region.reindex(sorted(by_region.index), axis=0)

display(by_day)
display(by_region)

In [ ]:
# Align by labels; do not assume a particular metro is absent.
common = by_region.index.intersection(by_day.columns)
by_region = by_region.loc[common]
by_day = by_day.loc[:, common]


In [ ]:
by_day.to_csv('google-trend-Coronavirus_disease_2019_byday_sorted.csv')
by_region.to_csv('google-trend-Coronavirus_disease_2019_byregion_sorted.csv')

## Combine

In [ ]:
by_region

In [ ]:
by_day.T

In [ ]:
q = by_region*by_day.T
max_ = q.max().max()
min_ = q.min().min()
q = (q-min_)/(max_-min_)

In [ ]:
q

In [ ]:
q.T.iloc[:,:5].plot(figsize=(15,5))